# M2.1 Gauss' Law on a Grid

Book pointer: chapter 4, sections 4.1 (motivation), 4.2 (integral form), 4.3 (differential form), 4.4 (proof).

## 1. Recall

1. Define electric flux through a surface. What does the sign of $\mathbf{E}\cdot d\mathbf{S}$ mean on a closed surface?
2. State Gauss' law in integral form. What is on the right side, and what is not?
3. Use the divergence theorem to go from the integral form to $\nabla\cdot\mathbf{E} = \rho/\epsilon_0$.
4. Why does the flux through a closed surface not depend on the shape of the surface or the position of the charge inside it?
5. What symmetry is needed to pull $|\mathbf{E}|$ out of the flux integral, and what three geometries have it?

## 2. Principle to algorithm

Flux through a closed surface is a sum over patches of $\mathbf{E}\cdot\hat{\mathbf{n}}\,\Delta S$. On a grid the natural closed surface is a box aligned with the axes. Each face has a constant normal, so the flux through a face is the integral of one field component over that face, and the trapezoid rule over the face samples does the integral. Six faces, signed by outward normal, give the total. The library function `fields.box_flux` does exactly that.

The differential form is tested with the same tools as module 1: `fields.divergence` of a gridded $\mathbf{E}$ should vanish wherever there is no charge, and the volume integral of the divergence over the box should equal the box flux.

### Animation

A point charge moves from the centre of a sphere to the edge and then outside. Arrow length and colour show $\mathbf{E}\cdot d\mathbf{S}$ on each patch. Inside, every patch is positive and the total is $q/\epsilon_0$ regardless of position. Outside, patches on the near side turn negative and the total cancels.

In [ ]:
from emlib import show
show.animation("m2_electrostatics", "GaussianSphereFlux")

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import grids, fields, relax
from emlib.fields import EPS0, K

plt.rcParams["figure.figsize"] = (6, 5)

In [ ]:
# A point charge in a 3D grid, and a box around it.
q = 1e-9
pos = np.array([[0.1, -0.2, 0.05]])
x, y, z, P = grids.grid3d((-1, 1), (-1, 1), (-1, 1), 81)
h = grids.spacing(x, y, z)
E = fields.point_charges_E(pos, [q], P)

def flux_through_cube(half_width):
    """Flux through the cube |x|,|y|,|z| <= half_width centred on the grid origin."""
    k = int(round(half_width / h[0]))
    c = 40  # index of the origin
    return fields.box_flux(E, h, (c - k, c - k, c - k), (c + k, c + k, c + k))

for w in (0.3, 0.5, 0.75, 1.0):
    f = flux_through_cube(w)
    print(f"cube half-width {w:.2f}:  flux / (q/eps0) = {f / (q / EPS0):.5f}")

The ratio is one for every box that contains the charge, independent of size and of where the charge sits inside. The small deviations are trapezoid-rule error on the faces, and they shrink with a finer grid.

### Exercise: flux with the charge outside

Replace the body of `flux_outside` so that it builds a cube that does not contain the charge and returns the flux through it. The check requires the result to be far smaller than $q/\epsilon_0$.

In [ ]:
def flux_outside():
    """Flux through a cube that excludes the charge at pos."""
    # the charge is near index (44, 32, 42); this cube lives in the corner x,y,z in [0.3, 1.0]
    return fields.box_flux(E, h, (52, 52, 52), (80, 80, 80))    # replace with your own box


f_out = flux_outside()
print(f"flux through a box with no charge inside / (q/eps0) = {f_out / (q / EPS0):.2e}")
assert abs(f_out) < 1e-2 * q / EPS0

In [ ]:
# Differential form. Away from the charge the divergence is zero to finite-difference accuracy.
div = fields.divergence(E, *h)
r = grids.magnitude(P - pos[0])
far = r > 0.25
print(f"max |div E| at least 0.25 from the charge: {np.abs(div[far]).max():.3e}  V/m^2")
print(f"typical |E| there:                            {grids.magnitude(E)[far].mean():.3e}  V/m")

plt.imshow(np.log10(np.abs(div[:, :, 42]) + 1e-3).T, origin="lower", extent=(-1, 1, -1, 1), cmap="magma")
plt.colorbar(label="log10 |div E| in the plane z = 0.05")
plt.title("the divergence is concentrated at the charge"); plt.show()

## 4. Verify

Three charges, one outside the box. The flux must equal the sum of the enclosed charges only. Then check that the volume integral of the numerical divergence over the same box equals the box flux, which is the divergence theorem on the grid.

In [ ]:
pos3 = np.array([[0.2, 0.1, 0.0], [-0.3, -0.2, 0.1], [0.0, 0.0, 1.6]])
q3 = np.array([2e-9, -1e-9, 5e-9])
E3 = fields.point_charges_E(pos3, q3, P)

flux3 = fields.box_flux(E3, h, (10, 10, 10), (70, 70, 70))     # cube |x|,|y|,|z| <= 0.75
enclosed = q3[:2].sum()
print(f"flux = {flux3:.4e}   enclosed charge / eps0 = {enclosed / EPS0:.4e}   ratio {flux3 / (enclosed / EPS0):.5f}")
assert abs(flux3 / (enclosed / EPS0) - 1) < 5e-3

div3 = fields.divergence(E3, *h)
box = (slice(10, 71),) * 3
vol_int = div3[box]
for ax in range(3):
    vol_int = np.trapezoid(vol_int, dx=h[ax], axis=0)
print(f"volume integral of div E over the same box = {vol_int:.4e}   (divergence theorem check)")

## 5. Explore and rebuild

Explore. Replace the point charges with a uniformly charged ball of radius $R$ built from a grid of small charges. Measure the flux through concentric cubes with half-width from $0.2R$ to $3R$ and plot flux against half-width. Identify the two regimes and compare the interior one to $Q_{\text{enc}}(r) = Q (r/R)^3$. What does a cube do to the clean $r^3$ law?

Rebuild. Close this notebook and write `box_flux` from scratch so it handles both 2D and 3D grids.